## **Eval of Finetuned Qwen3-8B with a Cross-Encoder**


In [8]:
%pip -q install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.5 MB/s eta 0:00:00


Miscellaneous

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

Define a function to load the two models (base and finetuned). Run one after the other.

In [10]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
    )
    return tokenizer, model

Define function to prompt

In [11]:
def prompt_model(tokenizer, model, prompt, max_new_tokens=500):
    messages = [
        {"role": "system", "content": "Rewrite the user's text in Gen-Z brainrot slang. Keep the meaning."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

Setting up evaluator (cross encoder)

In [13]:
from sentence_transformers import util, CrossEncoder # SentenceTransformer

# evaluator = SentenceTransformer("all-MiniLM-L6-v2")
evaluator = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

def semantic_score(reference, prediction):
    score = evaluator.predict([reference, prediction])
    return score.item()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Read test lines

In [14]:
import json
data = []
with open("test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

Run base and finetuned model on test prompts and compare to test answers

In [15]:
tokenizer, base_model = load_model("Qwen/Qwen3-4B")
base_answers = [prompt_model(
    tokenizer,
    base_model,
    item["source"]
) for item in data]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
finetuned_tokenizer, finetuned_model = load_model("PrimitivePenguin/brainrot-qwen3-4b-GGUF")
finetuned_answers = [prompt_model(
    finetuned_tokenizer,
    finetuned_model,
    item["source"]
) for item in data]

In [17]:
base_scores = [semantic_score(
    base_answers[i], data[i]["target"]
) for i in range(len(data))]
finetuned_scores = [semantic_score(
    finetuned_answers[i], data[i]["target"]
) for i in range(len(data))]


[-4.280340194702148, -6.42253303527832, -8.259696960449219, -4.490542411804199, -9.94927978515625, -11.031106948852539, -9.968727111816406, -8.383161544799805, -3.588351249694824, -4.766079425811768, -5.2515716552734375, -6.467937469482422, 6.382845878601074, -4.4225006103515625, -3.9777328968048096, -9.248188018798828, 0.2932406961917877, -4.726839542388916, 2.897148847579956, -6.226431369781494, -7.616680145263672, -7.968881130218506, -5.471635341644287, 0.2016385942697525, -11.392807960510254, -10.052680969238281, -11.370509147644043, 5.809130668640137, -10.536735534667969, -0.37030911445617676, -10.679826736450195, -11.017242431640625, 0.11561748385429382, -10.308906555175781, -1.0006110668182373]
